In [1]:
from google.colab import files
uploaded = files.upload()



Saving cora.cites to cora.cites


In [2]:
import networkx as nx

edge_file = "cora.cites"

G = nx.Graph()   # undirected graph (important)

with open(edge_file, "r") as f:
    for line in f:
        u, v = line.strip().split()
        G.add_edge(u, v)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Nodes: 2708
Edges: 5278


In [3]:
import random

edges = list(G.edges())
random.seed(42)
random.shuffle(edges)

num_test = int(0.1 * len(edges))   # 10% for test
test_pos = edges[:num_test]
train_edges = edges[num_test:]


In [4]:
G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

print("Train edges:", G_train.number_of_edges())


Train edges: 4751


In [5]:
non_edges = list(nx.non_edges(G))
random.shuffle(non_edges)

test_neg = non_edges[:num_test]


In [6]:
test_edges = [(u, v, 1) for (u, v) in test_pos] + \
             [(u, v, 0) for (u, v) in test_neg]

print("Test samples:", len(test_edges))


Test samples: 1054


In [7]:
def cn_score(G, u, v):
    return len(list(nx.common_neighbors(G, u, v)))


In [8]:
import math

def aa_score(G, u, v):
    score = 0
    for z in nx.common_neighbors(G, u, v):
        deg = G.degree(z)
        if deg > 1:
            score += 1 / math.log(deg)
    return score


In [9]:
def ra_score(G, u, v):
    score = 0
    for z in nx.common_neighbors(G, u, v):
        score += 1 / G.degree(z)
    return score


In [10]:
def sp_score(G, u, v):
    try:
        return 1 / nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        return 0


In [11]:
from sklearn.metrics import average_precision_score


In [12]:
def evaluate(method_name, score_func):
    y_true = []
    y_score = []

    for u, v, label in test_edges:
        y_true.append(label)
        y_score.append(score_func(G_train, u, v))

    ap = average_precision_score(y_true, y_score)
    print(f"{method_name} AP: {ap:.4f}")


In [13]:
evaluate("CN", cn_score)
evaluate("AA", aa_score)
evaluate("RA", ra_score)
evaluate("SP", sp_score)



CN AP: 0.7067
AA AP: 0.7182
RA AP: 0.7190
SP AP: 0.8443


In [14]:
# ===== Katz with AP evaluation =====

import numpy as np
from sklearn.metrics import average_precision_score

# Build adjacency matrix from training graph
nodes = list(G_train.nodes())
node_index = {n: i for i, n in enumerate(nodes)}

A = nx.to_numpy_array(G_train, nodelist=nodes)

# Katz parameters
beta = 0.005
L = 3   # path length considered

# Katz computation
K = np.zeros_like(A)

A_power = A.copy()
for l in range(1, L + 1):
    K += (beta ** l) * A_power
    A_power = A_power @ A

# Katz score function
def katz_score(G, u, v):
    return K[node_index[u], node_index[v]]

# Evaluate AP
y_true = []
y_score = []

for u, v, label in test_edges:
    y_true.append(label)
    y_score.append(katz_score(G_train, u, v))

print("Katz AP:", average_precision_score(y_true, y_score))


Katz AP: 0.8037558802444187
